In [1]:
import pandas as pd
import numpy as np

file_path = '../data/raw/BGL.log' # Update extension if it is .txt

parsed_logs = []
limit = 100000 # Loading 100k lines initially to avoid memory crashes on standard laptops

print("Parsing raw BGL logs...")
with open(file_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= limit:
            break
        
        parts = line.strip().split()
        if not parts:
            continue
            
        # Column 0 is the label ('-' is normal, anything else is an anomaly)
        label_str = parts[0]
        label = 0 if label_str == '-' else 1
        
        # Column 1 is the timestamp (useful for ordering)
        timestamp = parts[1]
        
        # The actual log message usually starts after the 9th space in standard BGL format
        # We join the remaining parts to get the core message
        message = " ".join(parts[9:]) if len(parts) > 9 else " ".join(parts[2:])
        
        parsed_logs.append([label, timestamp, message])

# Convert to a structured DataFrame
bgl_df = pd.DataFrame(parsed_logs, columns=['Label', 'Timestamp', 'Message'])

print(f"Parsed {len(bgl_df)} logs.")
display(bgl_df.head())
print("\nLabel Distribution:")
print(bgl_df['Label'].value_counts())

Parsing raw BGL logs...
Parsed 100000 logs.


,Label,Timestamp,Message
0,0,1117838570,instruction cache parity error corrected
1,0,1117838570,instruction cache parity error corrected
2,0,1117838570,instruction cache parity error corrected
3,0,1117838570,instruction cache parity error corrected
4,0,1117838570,instruction cache parity error corrected



Label Distribution:
Label
0    97391
1     2609
Name: count, dtype: int64


In [2]:
import re

def clean_log_message(msg):
    # Remove IP addresses
    msg = re.sub(r'\d+\.\d+\.\d+\.\d+', '<IP>', msg)
    # Remove hex numbers (like memory addresses)
    msg = re.sub(r'0x[a-fA-F0-9]+', '<HEX>', msg)
    # Remove standard numbers
    msg = re.sub(r'\b\d+\b', '<NUM>', msg)
    return msg.strip()

bgl_df['Cleaned_Message'] = bgl_df['Message'].apply(clean_log_message)

display(bgl_df[['Message', 'Cleaned_Message']].head())

,Message,Cleaned_Message
0,instruction cache parity error corrected,instruction cache parity error corrected
1,instruction cache parity error corrected,instruction cache parity error corrected
2,instruction cache parity error corrected,instruction cache parity error corrected
3,instruction cache parity error corrected,instruction cache parity error corrected
4,instruction cache parity error corrected,instruction cache parity error corrected


In [3]:
window_size = 20
sequences = []
sequence_labels = []

# Map each unique cleaned log message to an Event ID
unique_events = bgl_df['Cleaned_Message'].unique()
event2idx = {event: idx + 1 for idx, event in enumerate(unique_events)}
bgl_df['Event_ID'] = bgl_df['Cleaned_Message'].map(event2idx)

event_ids = bgl_df['Event_ID'].values
labels = bgl_df['Label'].values

# Create sliding windows
for i in range(len(event_ids) - window_size):
    seq = event_ids[i : i + window_size]
    # If any log in the window is an anomaly, tag the whole window as an anomaly
    seq_label = 1 if sum(labels[i : i + window_size]) > 0 else 0
    
    sequences.append(seq)
    sequence_labels.append(seq_label)

X_bgl = np.array(sequences)
y_bgl = np.array(sequence_labels)

print(f"Total Sequences: {len(X_bgl)}")
print(f"Sequence Shape: {X_bgl.shape}")
print(f"Label Distribution in Sequences:\n{pd.Series(y_bgl).value_counts()}")

Total Sequences: 99980
Sequence Shape: (99980, 20)
Label Distribution in Sequences:
0    89618
1    10362
Name: count, dtype: int64
